In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1424_Jawaharlal_Nehru_Stadium_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,164.93,244.49,7.12,19.97,27.07,50.01,5.78,0.99,10.23,...,NaN,10.44,75.72,1.06,271.26,0.0,0.0,45.33,987.12,NaN
1,2024-01-02,175.93,269.10,32.91,25.28,58.18,47.15,7.51,1.07,10.75,...,NaN,9.75,73.61,1.01,272.94,0.0,0.0,46.11,986.60,NaN
2,2024-01-03,175.23,259.61,26.86,41.62,49.93,49.87,10.82,1.47,13.11,...,NaN,9.47,84.85,0.92,218.29,0.0,0.0,41.88,986.11,NaN
3,2024-01-04,211.79,313.17,21.02,57.30,47.72,45.61,16.41,1.44,13.54,...,NaN,9.65,85.03,1.20,NaN,0.0,0.0,25.84,984.64,NaN
4,2024-01-05,165.32,268.59,19.67,57.63,46.84,50.54,18.02,1.32,10.27,...,NaN,10.53,87.98,1.28,197.92,0.0,0.0,19.48,983.99,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,276.56,391.22,215.57,84.09,220.01,77.31,5.64,3.72,13.00,...,NaN,16.88,76.50,1.66,164.50,0.0,0.0,5.00,992.67,NaN
362,2024-12-28,117.93,165.44,26.07,67.72,54.91,82.85,15.96,1.27,11.20,...,NaN,17.66,81.35,1.23,121.27,0.0,0.0,21.80,992.31,NaN
363,2024-12-29,87.40,124.59,10.09,41.82,30.40,59.24,6.50,0.73,21.59,...,NaN,16.77,79.22,1.26,227.14,0.0,0.0,54.62,995.02,NaN
364,2024-12-30,88.38,127.49,12.40,45.00,34.01,44.05,6.67,0.80,29.38,...,NaN,15.23,76.65,1.19,242.65,0.0,0.0,49.77,992.26,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         164.93        244.49        7.12        19.97   
1  2024-01-02         175.93        269.10       32.91        25.28   
2  2024-01-03         175.23        259.61       26.86        41.62   
3  2024-01-04         211.79        313.17       21.02        57.30   
4  2024-01-05         165.32        268.59       19.67        57.63   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      27.07        50.01         5.78        0.99          10.23   
1      58.18        47.15         7.51        1.07          10.75   
2      49.93        49.87        10.82        1.47          13.11   
3      47.72        45.61        16.41        1.44          13.54   
4      46.84        50.54        18.02        1.32          10.27   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             1.50             3.31    10.44   75.72      1

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.633116,0.713811,-0.799106,-1.607511,-0.801242,1.537438,-1.069204,-0.623426,-1.297451,-0.377288,-0.947188,-1.958675,1.036297,-1.254709,1.120629,0.0,0.0,-1.456842,0.678989
1,2024-01-02,1.825215,0.968504,0.115837,-1.371192,0.065624,1.247545,-0.785083,-0.455959,-1.274023,-0.357492,-0.950487,-2.042874,0.886474,-1.589078,1.145745,0.0,0.0,-1.440139,0.600439
2,2024-01-03,1.812990,0.870290,-0.098797,-0.643989,-0.164258,1.523247,-0.241475,0.381377,-1.167700,1.311935,-0.292302,-2.077041,1.684582,-2.190944,0.328709,0.0,0.0,-1.530718,0.526421
3,2024-01-04,2.451457,1.424591,-0.305980,0.053841,-0.225839,1.091448,0.676582,0.318576,-1.148328,2.790005,1.184080,-2.055076,1.697363,-0.318473,-0.143422,0.0,0.0,-1.874191,0.304365
4,2024-01-05,1.639927,0.963225,-0.353874,0.068527,-0.250360,1.591160,0.940996,0.067376,-1.295648,2.130153,1.182430,-1.947693,1.906831,0.216518,0.024171,0.0,0.0,-2.010381,0.206178
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,-0.318346,2.232342,-0.246557,1.246115,-0.223192,-0.161885,-1.092197,-0.267558,-1.172656,2.268722,2.365184,-1.172826,1.091681,2.757728,-0.475469,0.0,0.0,-2.320449,1.517361
362,2024-12-28,0.812330,-0.104290,-0.126823,0.517577,-0.025493,-0.161885,0.602678,-0.037291,-1.253750,-0.251916,0.225669,-1.077646,1.436061,-0.117852,-1.121772,0.0,0.0,-1.960702,1.462981
363,2024-12-29,0.279169,-0.527053,-0.693740,-0.635089,-0.708453,2.473003,-0.950957,-1.167694,-0.785657,-0.839185,-0.889452,-1.186249,1.284818,0.082770,0.461020,0.0,0.0,-1.257910,1.872348
364,2024-12-30,0.296283,-0.497040,-0.611789,-0.493564,-0.607862,0.933325,-0.923038,-1.021160,-0.434700,-0.799594,-0.874606,-1.374169,1.102332,-0.385347,0.692900,0.0,0.0,-1.361766,1.455428


In [10]:
df.to_excel('jawaharlalstadium2024.xlsx', index=False)